In [ ]:
import os, re, subprocess, glob, time, json
import numpy as np
t0 = time.time()
OUT = '/kaggle/working/repo'
os.makedirs(OUT, exist_ok=True)
os.chdir(OUT)

print('в /kaggle/input:', os.listdir('/kaggle/input'), flush=True)
hits = sorted(glob.glob('/kaggle/input/**/L400/*/X.dat', recursive=True))
SRC = {os.path.basename(os.path.dirname(h)): os.path.dirname(h) for h in hits}
print('нашлись наборы:', sorted(SRC), flush=True)
for k in sorted(SRC):
    meta = json.load(open(SRC[k] + '/meta.json'))
    y = np.memmap(SRC[k] + '/y.dat', dtype='float32', mode='r', shape=(meta['N'], 3))
    print(f"  {k}: окон {meta['N']}, людей {len(np.unique(y[:,1]))}", flush=True)

# набор берём как есть
os.makedirs('Processed', exist_ok=True)
link = 'Processed/ADFTD-PS'
if not os.path.exists(link):
    os.symlink(SRC['ADFTD-PS'], link)
ROOT = './Processed'

subprocess.run('pip install -q huggingface_hub einops', shell=True)
subprocess.run('git clone -q https://github.com/DL4mHealth/LEAD.git', shell=True)

# В adftd_loader.py две функции get_id_list_adftd, побеждает вторая (memmap),
# и ветки 5-fold в ней нет. Переносим её из первой функции того же файла.
P = 'LEAD/data_provider/dataset_loader/adftd_loader.py'
s = open(P).read()
OLD = """    else:
        raise ValueError('Invalid cross_val. Please use fixed, mccv, or loso.')


class ADFTDLoader(BaseLoader):"""
NEW = """    elif args.cross_val == '5-fold':
        random.seed(42)
        random.shuffle(hc_list)
        random.shuffle(ad_list)
        random.shuffle(ftd_list)
        fh, fa, ff = len(hc_list) / 5, len(ad_list) / 5, len(ftd_list) / 5
        k = (args.seed - 41) % 5
        test_ids = (hc_list[int(k * fh):int((k + 1) * fh)] +
                    ad_list[int(k * fa):int((k + 1) * fa)] +
                    ftd_list[int(k * ff):int((k + 1) * ff)])
        if k == 4:
            val_ids = hc_list[:int(fh)] + ad_list[:int(fa)] + ftd_list[:int(ff)]
        else:
            val_ids = (hc_list[int((k + 1) * fh):int((k + 2) * fh)] +
                       ad_list[int((k + 1) * fa):int((k + 2) * fa)] +
                       ftd_list[int((k + 1) * ff):int((k + 2) * ff)])
        train_ids = [sid for sid in all_ids
                     if sid not in test_ids and sid not in val_ids]
        return sorted(all_ids), sorted(train_ids), sorted(val_ids), sorted(test_ids)

    else:
        raise ValueError('Invalid cross_val. Please use fixed, mccv, 5-fold, or loso.')


class ADFTDLoader(BaseLoader):"""
assert s.count(OLD) == 1
open(P, 'w').write(s.replace(OLD, NEW))

CK = 'checkpoints/LEAD/pretrain/LEAD/P-11-b1024-p50-g16-learnable-3D/nh8_el12_dm128_df256_seed41/checkpoint.pth'
os.makedirs(os.path.dirname(CK), exist_ok=True)
from huggingface_hub import hf_hub_download
hf_hub_download(repo_id='YiheWang/LEAD', filename=CK, local_dir='.')

LOG = '/kaggle/working/lead_ps-adhc.log'
CMD = (
    'python LEAD/run.py --method LEAD --task_name finetune --is_training 1 '
    '--root_path ' + ROOT + ' --data_path L400 --data MultiDatasets '
    '--training_dataset ADFTD-PS --model LEAD '
    '--model_id P-11-F-ps-adhc '
    '--checkpoints_path ./' + CK + ' '
    '--e_layers 12 --batch_size 64 --n_heads 8 --d_model 128 --d_ff 256 '
    '--patch_len 50 --stride 50 --temporal_pos_type learnable --channel_pos_type 3D '
    '--use_sampling_embedding --sampling_rate_list 200,100,50 '
    '--group_shuffle --group_size 4 --use_subject_loss --use_subject_vote '
    '--classify_choice ad_vs_hc --cross_val 5-fold --itr 5 '
    '--train_epochs 8 --patience 3 --learning_rate 0.0001 '
    '--seq_len 400 --enc_in 19 --des 5fold_ps-adhc '
    '; exit ${PIPESTATUS[0]}'
)
print(CMD, flush=True)
with open(LOG, 'w') as fh:
    p = subprocess.Popen(CMD, shell=True, executable='/bin/bash',
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        fh.write(line); fh.flush()
        if re.search(r'Epoch|results|Error|Traceback|subjects', line):
            print(line.rstrip()[:260], flush=True)
rc = p.wait()
print('return code', rc, '| elapsed', round((time.time() - t0) / 60, 1), 'min', flush=True)
if rc != 0:
    raise RuntimeError('LEAD run.py завершился с кодом %d' % rc)
